In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.feature import BORDERS, LAND, OCEAN
from ipywidgets import interact, IntSlider, Dropdown

from ml_baselines.features import open_features
from ml_baselines.config import Config

cfg = Config()

In [ ]:
site = "MHD"
df_features = open_features(site, 2000, 2001, time_shift_hours=[3,6,])

print("total features: ", len(df_features.columns))

In [ ]:
def variables_points_timesteps(columns):
    """Parse the column names to extract variables, points, and timesteps.
    Assumes column names are in the format "variable_point_timestep" or "variable_point" or "variable" (for 0D variables).
    
    Args:
        columns (list): List of column names.

    Returns:
        tuple: A tuple containing lists of 0D variables, 1D variables, points, and timesteps.
    """


    variables_0d = ["hour_of_day", "day_of_year"]
    variables = []
    points = []
    timesteps = []

    for col in columns:
        if col in variables_0d:
            continue
        elif len(col.split("_")) == 3:
            var, point, timestep = col.split("_")
            variables.append(var)
            points.append(point)
            timesteps.append(timestep)
        elif len(col.split("_")) == 2:
            var, point = col.split("_")
            variables.append(var)
            points.append(point)
            timesteps.append("")

    if not len(variables) + len(variables_0d) == len(columns):
        raise ValueError("Have you accounted for all of the zero-dimension variables?")

    variables = sorted(list(set(variables)))
    points = sorted([int(p) for p in list(set(points))])
    timesteps = sorted(list(set(timesteps)))

    return variables_0d, variables, points, timesteps

In [ ]:
variables_0d, variables, points, timesteps = variables_points_timesteps(df_features.columns)
print("Variables (0D):", variables_0d)
print("Variables:", variables)
print("Points:", points)
print("Timesteps:", timesteps)

In [ ]:
# Timeseries figure, with selector for the variable to plot
def plot_timeseries(met_variable, point, lag):

    if met_variable in variables_0d:
        variable = met_variable
    else:
        variable = f"{met_variable}_{point}_{lag}" if lag else f"{met_variable}_{point}"
    plt.figure(figsize=(10, 5))
    plt.plot(df_features.index, df_features[variable], label=variable)
    plt.xlabel("Time")
    plt.ylabel(variable)
    plt.title(f"{variable} at {site}")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
# Interactive widget to select variable
point = IntSlider(min=points[0], max=points[-1], step=1, value=0)
met_variable = Dropdown(options=variables_0d + variables, description='Variable:')
time_lag = Dropdown(options=timesteps, description='Time Lag:')

interact(plot_timeseries, met_variable=met_variable, point=point, lag=time_lag)

In [ ]:
def interactive_wind_plot(time_idx=0):

    wind_variables=["u10", "v10"]
    
    # Get the grid coordinates
    lats = cfg.lats_grid + cfg.site_coords_dict[site][0]
    lons = cfg.lons_grid + cfg.site_coords_dict[site][1]
    
    # Extract u and v components
    u_components = [df_features.iloc[time_idx][f"{wind_variables[0]}_{i}"] for i in range(17)]
    v_components = [df_features.iloc[time_idx][f"{wind_variables[1]}_{i}"] for i in range(17)]
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Plot wind vectors
    quiver = ax.quiver(lons, lats, u_components, v_components, 
                       scale=100, scale_units='width', width=0.003,
                       color='blue', alpha=0.7)
    
    # Add reference arrow
    ax.quiverkey(quiver, 0.9, 0.95, 10, '10 m/s', labelpos='E')
    
    # Customize the plot
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude') 
    ax.set_title(f'Wind Vectors at Grid Points (Time: {df_features.index[time_idx]})')
    ax.grid(True, alpha=0.3)
    
    # Add grid point markers
    ax.scatter(lons, lats, c='red', s=30, zorder=5, alpha=0.8)
    
    # Add map background
    ax.coastlines()
    ax.add_feature(BORDERS, linestyle=':')
    ax.add_feature(LAND, edgecolor='black', alpha=0.5)
    ax.add_feature(OCEAN, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create interactive widget
time_slider = IntSlider(min=0, max=len(df_features)-1, step=1, value=0, description='Time:')
interact(interactive_wind_plot, time_idx=time_slider)